# Bruker timsTOF → imzML Converter
### MALDI Imaging · non-PASEF QTOF · TSF format

Converts a Bruker `.d` directory (`analysis.tsf`) to the open **imzML + .ibd** format, compatible with Cardinal, MSiReader, SCiLS, and other MSI tools.

---

## How to use this notebook

| Step | Cell | Action |
|------|------|--------|
| 1 | **Cell 1** | Install Python dependencies |
| 2 | **Cell 2** | Check SDK library availability |
| 3 | **Cell 3** | ⚙️ **Set your paths & options** ← only cell you edit |
| 4 | **Cell 4** | Load converter code |
| 5 | **Cell 5** | Run the conversion |
| 6 | **Cell 6** | Verify output files |

---

## Format background

**Bruker `.d` input** contains two key files:

| File | Contents |
|------|----------|
| `analysis.tsf` | SQLite database — metadata (frames, calibration, XY stage positions) |
| `analysis.tsf_bin` | Binary blob — raw spectra, read-only via Bruker SDK |

**imzML output** is also two files:

| File | Contents |
|------|----------|
| `.imzML` | XML — pixel coordinates, byte offsets into `.ibd`, CV-term annotations |
| `.ibd` | Binary — all m/z + intensity arrays written end-to-end |

> **Processed mode** (used here): each pixel has its own m/z axis — correct for
> centroided MALDI data where peak lists differ between pixels.

---
## Cell 1 — Install Python dependencies

`numpy` is required. `tqdm` adds a live progress bar during conversion.  
Run once; skip on subsequent runs.

In [1]:
%pip install numpy tqdm

Note: you may need to restart the kernel to use updated packages.


---
## Cell 2 — Bruker SDK check

The converter calls the **Bruker timsdata SDK** (a compiled C library) to decode raw spectral data.

**Option A — set `SDK_PATH` in Cell 3** *(recommended)*  
Point it at the full `.dll` / `.so` path inside the TDF-SDK zip:
```
timsdata/win64/timsdata.dll        ← Windows
timsdata/linux64/libtimsdata.so    ← Linux
```

**Option B — put the library on your system path**  
- Windows: copy `timsdata.dll` to the working directory or add its folder to `PATH`
- Linux: add the folder to `LD_LIBRARY_PATH`, or copy it to `/usr/local/lib`

Run this cell to confirm which library name Python will look for on your platform.

In [2]:
import sys

libname = 'timsdata.dll' if sys.platform.startswith('win') else 'libtimsdata.so'

print(f'Platform         : {sys.platform}')
print(f'Expected library : {libname}')
print()
print('If Cell 5 raises an OSError, set SDK_PATH in Cell 3 to the full path of this file.')

Platform         : win32
Expected library : timsdata.dll

If Cell 5 raises an OSError, set SDK_PATH in Cell 3 to the full path of this file.


---
## Cell 3 — ⚙️ Configuration  *(edit this cell)*

This is the **only cell you need to modify**.

In [3]:
# ── REQUIRED ────────────────────────────────────────────────────────────────

# Full path to your Bruker .d directory
BRUKER_D_PATH = r"C:\Users\maddi\Documents\ColleyVUApps\QTOFBrain\RatBrainNeg.d"

# Where to write the output .imzML and .ibd files
OUTPUT_DIR = r"C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs"

# Base name for output files (no extension)
# → OUTPUT_DIR/OUTPUT_NAME.imzML  and  OUTPUT_DIR/OUTPUT_NAME.ibd
OUTPUT_NAME = 'RatBrainNegConverted'

# ── SDK PATH ────────────────────────────────────────────────────────────────

# Full path to timsdata.dll (Windows) or libtimsdata.so (Linux).
# Set to None if the library is already on PATH / LD_LIBRARY_PATH (Option B).

SDK_PATH = r'C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\timsdata\win64\timsdata.dll'
# SDK_PATH = '/path/to/timsdata/linux64/libtimsdata.so'      # Linux example

# ── CONVERSION OPTIONS ──────────────────────────────────────────────────────

# 'centroid' → peak-picked spectra (smaller files, recommended for most analyses)
# 'profile'  → raw profile spectra (much larger, preserves all raw signal)
SPECTRUM_TYPE = 'centroid'

# True  → use post-acquisition recalibration from DataAnalysis if available (recommended)
# False → use original acquisition-time calibration only
USE_RECALIBRATED = True

# MS level filter:
#   0    → MS1 frames only (standard for MALDI imaging)
#   2    → MS2 frames only
#   None → export all frames regardless of level
MS_LEVEL_FILTER = 0

# ── CONFIRM SETTINGS ────────────────────────────────────────────────────────
print('=== Conversion settings ===')
print(f'  Input  : {BRUKER_D_PATH}')
print(f'  Output : {OUTPUT_DIR}/{OUTPUT_NAME}.imzML')
print(f'  SDK    : {SDK_PATH or "(system path)"}')
print(f'  Type   : {SPECTRUM_TYPE}')
print(f'  Recal  : {USE_RECALIBRATED}')
print(f'  Level  : {"All" if MS_LEVEL_FILTER is None else "MS" + str(MS_LEVEL_FILTER)}')

=== Conversion settings ===
  Input  : C:\Users\maddi\Documents\ColleyVUApps\QTOFBrain\RatBrainNeg.d
  Output : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs/RatBrainNegConverted.imzML
  SDK    : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\timsdata\win64\timsdata.dll
  Type   : centroid
  Recal  : True
  Level  : MS0


---
## Cell 4 — Load converter code

Run this cell to define all functions and classes. **No need to edit it.**  
Inline annotations explain every non-obvious decision.

In [6]:
# ============================================================
# IMPORTS
# ctypes is the bridge from Python to the Bruker C SDK.
# It lets us call compiled C functions by manually declaring
# each function's argument and return types.
# ============================================================
import os, sys, uuid, hashlib, sqlite3, datetime
import numpy as np
from pathlib import Path
from ctypes import (
    cdll,               # loads a compiled shared library (.dll / .so)
    c_char_p,           # C type: char*   (UTF-8 string pointer)
    c_uint32,           # C type: unsigned 32-bit integer
    c_uint64,           # C type: unsigned 64-bit integer  <- SDK 'handle'
    c_int32,            # C type: signed 32-bit integer
    c_int64,            # C type: signed 64-bit integer    <- frame IDs
    c_float,            # C type: 32-bit float             <- intensities
    c_double,           # C type: 64-bit double            <- m/z values
    POINTER,            # builds a pointer type, e.g. POINTER(c_double) = double*
    create_string_buffer,  # allocates a mutable byte buffer for C to write into
)

# tqdm is optional — provides a progress bar during conversion.
# HAS_TQDM is always defined so _write_ibd() can safely reference it
# regardless of whether tqdm is installed.
# We use plain tqdm (not tqdm.notebook) to avoid the ipywidgets dependency —
# tqdm prints a clean text bar in Jupyter without needing any widgets.
HAS_TQDM = False
tqdm = None

try:
    from tqdm import tqdm   # plain text bar — works in Jupyter and terminal, no ipywidgets needed
    HAS_TQDM = True
except ImportError:
    print('tqdm not found — no progress bar. Install with: pip install tqdm')

print('Imports OK')

# ============================================================
# SECTION 1 — SDK LOADER
#
# The Bruker SDK is a compiled C library. We use ctypes to:
#   1. Load it into memory (cdll.LoadLibrary)
#   2. Declare argtypes + restype for every function we call.
#      Without this, ctypes cannot correctly convert Python
#      values to C types (a Python int defaults to 32-bit,
#      which would break 64-bit handle arguments).
# ============================================================

def _load_sdk(sdk_path=None):
    """Load the Bruker timsdata shared library and bind all TSF API functions."""

    if sdk_path is not None:
        lib = cdll.LoadLibrary(str(sdk_path))
    else:
        # Let the OS search its default library paths.
        # Windows: PATH + working dir.  Linux: LD_LIBRARY_PATH + /usr/lib.
        libname = 'timsdata.dll' if sys.platform.startswith('win') else 'libtimsdata.so'
        lib = cdll.LoadLibrary(libname)

    # tsf_open: opens a .d directory; returns an opaque integer 'handle'.
    # Every subsequent SDK call takes this handle as its first argument.
    # Returns 0 on failure.
    lib.tsf_open.argtypes = [c_char_p, c_uint32]  # (path_utf8, use_recalibrated)
    lib.tsf_open.restype  = c_uint64               # handle (0 = failure)

    # tsf_close: releases all resources. Always call — even on error paths.
    lib.tsf_close.argtypes = [c_uint64]
    lib.tsf_close.restype  = None

    # tsf_get_last_error_string: two-pass pattern for variable-length strings.
    # Call with (None, 0) to get the required buffer size, then call again
    # with a real buffer of that size to get the actual message.
    lib.tsf_get_last_error_string.argtypes = [c_char_p, c_uint32]
    lib.tsf_get_last_error_string.restype  = c_uint32  # required buffer length

    # tsf_has_recalibrated_state: returns 1 if post-acquisition recalibration
    # data is present (e.g. from Bruker DataAnalysis software).
    lib.tsf_has_recalibrated_state.argtypes = [c_uint64]
    lib.tsf_has_recalibrated_state.restype  = c_uint32

    # tsf_read_line_spectrum_v2: reads a peak-picked (centroided) spectrum.
    # Returns bin indices in index_array — NOT m/z values.
    # Bin indices must be converted to m/z with tsf_index_to_mz separately.
    # Returns -1 on error, or the peak count (may exceed 'length' if buffer
    # is too small — in that case, grow the buffer and retry).
    lib.tsf_read_line_spectrum_v2.argtypes = [
        c_uint64,           # handle
        c_int64,            # spectrum_id  (= Frames.Id in the SQLite DB)
        POINTER(c_double),  # index_array  (caller-allocated; SDK writes bin indices here)
        POINTER(c_float),   # intensity_array (caller-allocated; SDK writes intensities here)
        c_int32             # length: capacity of the arrays, in elements
    ]
    lib.tsf_read_line_spectrum_v2.restype = c_int32  # peak count, or -1

    # tsf_read_profile_spectrum_v2: reads a raw profile spectrum.
    # Returns a dense intensity array: position i = intensity at bin i.
    # No separate index array — all bins 0..N-1 are present.
    lib.tsf_read_profile_spectrum_v2.argtypes = [
        c_uint64,           # handle
        c_int64,            # spectrum_id
        POINTER(c_uint32),  # profile_array (uint32 intensities, one per bin)
        c_int32             # length: capacity of the array
    ]
    lib.tsf_read_profile_spectrum_v2.restype = c_int32  # profile length, or -1

    # tsf_index_to_mz: converts bin indices -> calibrated m/z values.
    # Calibration is per-frame (different polynomial per frame, stored in
    # CalibrationInfo). Always use this — never reconstruct it from SQLite.
    # Vectorised: pass N indices, get N m/z values in one call.
    lib.tsf_index_to_mz.argtypes = [
        c_uint64,           # handle
        c_int64,            # frame_id (calibration is per-frame)
        POINTER(c_double),  # in:  array of bin indices
        POINTER(c_double),  # out: array of m/z values (caller-allocated)
        c_uint32            # cnt: number of values to convert
    ]
    lib.tsf_index_to_mz.restype = c_uint32  # 1 on success, 0 on failure

    return lib


def _tsf_last_error(lib):
    """Fetch the SDK's last error message as a Python string."""
    n   = lib.tsf_get_last_error_string(None, 0)   # pass 1: get required length
    buf = create_string_buffer(n)                   # allocate buffer
    lib.tsf_get_last_error_string(buf, n)           # pass 2: fill buffer
    return buf.value.decode('utf-8', errors='replace')


print('SDK loader defined')


# ============================================================
# SECTION 2 — SPECTRUM READERS
#
# Both functions use a 'buffer-growing loop':
#   1. Allocate a numpy array of a guessed capacity.
#   2. Pass a raw C pointer into it — zero copy.
#   3. If the SDK needed more space, grow and retry.
#      Converges in <=2 iterations for almost all spectra.
# ============================================================

def _read_centroid(lib, handle, frame_id, buf_size=4096):
    """
    Read a peak-picked centroid spectrum. Returns (mz_array, intensity_array).

    The SDK returns floating-point bin indices (not integers) because it
    interpolates peak centres within a bin for sub-bin precision.
    We convert to physical m/z via tsf_index_to_mz.

    mzs  : float64 (64-bit; high precision needed for mass accuracy)
    ints : float32 (32-bit; sufficient for peak intensities)
    """
    while True:
        # np.empty skips zeroing — safe here because every element will be
        # overwritten by the SDK before we read it.
        idx_buf = np.empty(buf_size, dtype=np.float64)
        int_buf = np.empty(buf_size, dtype=np.float32)

        # .ctypes.data_as(POINTER(...)) gives the SDK a raw C pointer into
        # the numpy array's memory without making a copy.
        n = lib.tsf_read_line_spectrum_v2(
            handle, frame_id,
            idx_buf.ctypes.data_as(POINTER(c_double)),
            int_buf.ctypes.data_as(POINTER(c_float)),
            buf_size,
        )

        if n < 0:
            raise RuntimeError(f'tsf_read_line_spectrum_v2 failed: {_tsf_last_error(lib)}')
        if n > buf_size:
            buf_size = n    # SDK tells us exactly how much space it needed
            continue
        if n == 0:
            return np.empty(0, np.float64), np.empty(0, np.float32)

        # Slice to valid elements and copy (decouples from the reusable buffer).
        indices = idx_buf[:n].copy()
        mzs     = np.empty(n, dtype=np.float64)

        ok = lib.tsf_index_to_mz(
            handle, frame_id,
            indices.ctypes.data_as(POINTER(c_double)),
            mzs.ctypes.data_as(POINTER(c_double)),
            n,
        )
        if not ok:
            raise RuntimeError(f'tsf_index_to_mz failed: {_tsf_last_error(lib)}')

        return mzs, int_buf[:n].copy()


def _read_profile(lib, handle, frame_id, buf_size=65536):
    """
    Read a raw profile spectrum. Returns (mz_array, intensity_array).

    Profile spectra are dense: every bin 0..N-1 is present, so bin indices
    are just arange(N). Profile spectra are much larger than centroid
    (~50k–200k bins vs ~hundreds of peaks), hence the larger default buf_size.

    mzs  : float64 (one m/z per bin)
    ints : float32 (cast from uint32 for storage consistency)
    """
    while True:
        int_buf = np.empty(buf_size, dtype=np.uint32)

        n = lib.tsf_read_profile_spectrum_v2(
            handle, frame_id,
            int_buf.ctypes.data_as(POINTER(c_uint32)),
            buf_size,
        )

        if n < 0:
            raise RuntimeError(f'tsf_read_profile_spectrum_v2 failed: {_tsf_last_error(lib)}')
        if n > buf_size:
            buf_size = n
            continue
        if n == 0:
            return np.empty(0, np.float64), np.empty(0, np.float32)

        indices = np.arange(n, dtype=np.float64)   # bins 0, 1, 2, ..., n-1
        mzs     = np.empty(n, dtype=np.float64)

        ok = lib.tsf_index_to_mz(
            handle, frame_id,
            indices.ctypes.data_as(POINTER(c_double)),
            mzs.ctypes.data_as(POINTER(c_double)),
            n,
        )
        if not ok:
            raise RuntimeError(f'tsf_index_to_mz failed: {_tsf_last_error(lib)}')

        return mzs, int_buf[:n].astype(np.float32)


print('Spectrum readers defined')


# ============================================================
# SECTION 3 — imzML CV TERM CONSTANTS
#
# imzML tags every metadata field with a 'controlled vocabulary'
# (CV) accession from an agreed ontology:
#   MS:xxxxxxx  — PSI-MS ontology (general mass spectrometry)
#   IMS:xxxxxxx — Imaging MS ontology (spatial imaging extensions)
#   UO:xxxxxxx  — Unit Ontology (physical units)
#
# Named constants prevent typos and make the XML-building readable.
# ============================================================

CV_MZ_ARRAY        = 'MS:1000514'   # array contains m/z values
CV_INTENSITY_ARRAY = 'MS:1000515'   # array contains intensity values
CV_64BIT_FLOAT     = 'MS:1000523'   # 8 bytes/element (m/z — high precision)
CV_32BIT_FLOAT     = 'MS:1000521'   # 4 bytes/element (intensities)
CV_NO_COMPRESSION  = 'MS:1000576'   # data stored uncompressed
CV_CENTROID        = 'MS:1000127'   # peak-picked data
CV_PROFILE         = 'MS:1000128'   # raw profile data
CV_POSITIVE        = 'MS:1000130'   # positive ion mode
CV_NEGATIVE        = 'MS:1000129'   # negative ion mode
CV_MALDI           = 'MS:1000075'   # matrix-assisted laser desorption ionisation
CV_PIXEL_SIZE_X    = 'IMS:1000046'  # physical pixel width  (µm)
CV_PIXEL_SIZE_Y    = 'IMS:1000047'  # physical pixel height (µm)
CV_MAX_COUNT_X     = 'IMS:1000042'  # total columns in pixel grid
CV_MAX_COUNT_Y     = 'IMS:1000043'  # total rows in pixel grid
CV_PIXEL_COORD_X   = 'IMS:1000050'  # x (column) index of this pixel, 1-based
CV_PIXEL_COORD_Y   = 'IMS:1000051'  # y (row) index of this pixel, 1-based
CV_EXTERNAL_DATA   = 'IMS:1000101'  # data lives in external .ibd file
CV_EXTERNAL_OFFSET = 'IMS:1000102'  # byte offset into .ibd where array starts
CV_EXTERNAL_LENGTH = 'IMS:1000103'  # number of elements (not bytes) in array
CV_PROCESSED_MODE  = 'IMS:1000031'  # each pixel has its own m/z axis
CV_MZ_MIN          = 'MS:1000528'   # lowest observed m/z in this spectrum
CV_MZ_MAX          = 'MS:1000527'   # highest observed m/z in this spectrum

print('CV constants defined')


# ============================================================
# SECTION 4 — SPOT NAME PARSER
#
# Older Bruker datasets store pixel positions as alphanumeric
# spot names rather than numeric XY indices.
# Converts the three most common formats to (x, y) int coords.
# ============================================================

def _parse_spot_name(spot):
    """
    Parse a Bruker spot name into (x, y) 1-based pixel coordinates.

    Supported formats:
      'R01C02'  -> row/column -> (x=col=2, y=row=1)  most common
      '0042'    -> sequential -> (x=42, y=1)
      'B3'      -> plate well -> (x=3, y=2)  A=1, B=2, ...
    """
    import re
    if spot is None:
        return (1, 1)
    spot = str(spot).strip()

    m = re.match(r'[Rr](\d+)[Cc](\d+)', spot)   # R##C## format
    if m:
        return (int(m.group(2)), int(m.group(1)))  # (col=x, row=y)

    m = re.match(r'^(\d+)$', spot)               # pure integer
    if m:
        return (int(m.group(1)), 1)

    m = re.match(r'([A-Za-z]+)(\d+)', spot)      # letter + number (plate well)
    if m:
        row = sum((ord(c.upper()) - 64) * (26**i)  # base-26: A=1, B=2, ..., AA=27
                  for i, c in enumerate(reversed(m.group(1))))
        return (int(m.group(2)), row)

    return (1, 1)   # unrecognised; return origin


print('Spot name parser defined')


# ============================================================
# SECTION 5 — MAIN CONVERTER CLASS
#
# Pipeline:
#   __init__         validate paths, set options
#   convert()        orchestrate all stages; always closes SDK handle
#   _load_metadata() read GlobalMetadata + MaldiFrameInfo from SQLite
#   _load_frames()   build pixel list with XY coords from MaldiFrameInfo
#   _write_ibd()     read every spectrum via SDK, write binary .ibd
#   _write_imzml()   write XML .imzML with byte-offset pointers into .ibd
# ============================================================

class BrukerToImzML:

    def __init__(self, bruker_d_path, output_dir=None, output_name=None,
                 sdk_path=None, spectrum_type='centroid',
                 use_recalibrated=True, ms_level_filter=0):

        self.d_path           = Path(bruker_d_path)
        self.output_dir       = Path(output_dir) if output_dir else self.d_path.parent
        self.output_name      = output_name or self.d_path.stem
        self.sdk_path         = sdk_path
        self.spectrum_type    = spectrum_type.lower()
        self.use_recalibrated = use_recalibrated
        self.ms_level_filter  = ms_level_filter

        if self.spectrum_type not in ('centroid', 'profile'):
            raise ValueError("spectrum_type must be 'centroid' or 'profile'")

        # TSF = non-PASEF QTOF imaging (no ion mobility).
        # TDF = timsTOF with TIMS — different format, not handled here.
        if not (self.d_path / 'analysis.tsf').exists():
            raise FileNotFoundError(
                f'No analysis.tsf in {self.d_path}.\n'
                'This converter is for non-PASEF QTOF imaging (TSF format).\n'
                'timsTOF with ion mobility uses analysis.tdf instead.'
            )

        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.imzml_path = self.output_dir / f'{self.output_name}.imzML'
        self.ibd_path   = self.output_dir / f'{self.output_name}.ibd'

    # ── Orchestrator ─────────────────────────────────────────

    def convert(self):
        """Run the full pipeline. Returns (imzml_path, ibd_path) as strings."""
        print(f'Opening: {self.d_path}')
        lib = _load_sdk(self.sdk_path)

        # tsf_open returns an opaque integer handle (like a file descriptor).
        # Pass 1 = use recalibrated state; 0 = raw acquisition calibration.
        handle = lib.tsf_open(str(self.d_path).encode('utf-8'),
                              1 if self.use_recalibrated else 0)
        if handle == 0:
            raise RuntimeError(f'Failed to open dataset: {_tsf_last_error(lib)}')

        recal = lib.tsf_has_recalibrated_state(handle)
        print(f'Recalibrated state: available={bool(recal)} | using={self.use_recalibrated and bool(recal)}')

        try:
            # SQLite and the SDK work independently:
            # SDK reads analysis.tsf_bin (binary); SQLite reads analysis.tsf (DB).
            conn   = sqlite3.connect(str(self.d_path / 'analysis.tsf'))
            meta   = self._load_metadata(conn)
            frames = self._load_frames(conn)
            conn.close()

            if not frames:
                raise RuntimeError('No matching frames — check MS_LEVEL_FILTER.')

            print(f'Spectra to convert : {len(frames)}')
            print(f'Spectrum type      : {self.spectrum_type}')

            spec_meta = self._write_ibd(lib, handle, frames)
            self._write_imzml(meta, frames, spec_meta)

        finally:
            lib.tsf_close(handle)   # always release, even on error

        print(f'\nDone!')
        print(f'  imzML : {self.imzml_path}')
        print(f'  ibd   : {self.ibd_path}')
        return str(self.imzml_path), str(self.ibd_path)

    # ── Stage 0: Metadata ────────────────────────────────────

    def _load_metadata(self, conn):
        """
        Read acquisition metadata from the analysis.tsf SQLite database.

        GlobalMetadata: instrument/acquisition settings as key-value rows
        (mass range, laser settings, software version, etc.).
        MaldiFrameInfo: per-frame stage geometry (laser spot width/height
        -> pixel size). Optional tables are read with broad try/except
        because they may not exist in all dataset versions.
        """
        meta = {}
        for key, val in conn.execute('SELECT Key, Value FROM GlobalMetadata'):
            meta[key] = val

        # Read first MaldiFrameInfo row to harvest laser/geometry metadata.
        # PRAGMA table_info returns column definitions: (cid, name, type, ...)
        try:
            cols = [r[1] for r in conn.execute('PRAGMA table_info(MaldiFrameInfo)')]
            if cols:
                first = conn.execute('SELECT * FROM MaldiFrameInfo LIMIT 1').fetchone()
                if first:
                    for col, val in zip(cols, first):
                        meta[f'MaldiFrameInfo.{col}'] = val
        except Exception:
            pass

        for table in ('ScanMode', 'MaldiApplicationInfo'):
            try:
                for key, val in conn.execute(f'SELECT Key, Value FROM {table}'):
                    meta[f'{table}.{key}'] = val
            except Exception:
                pass

        return meta

    # ── Stage 1: Frame list ──────────────────────────────────

    def _load_frames(self, conn):
        """
        Build the list of pixels to convert from Frames + MaldiFrameInfo.

        Each Bruker frame = one laser shot = one pixel in the image.
        We need: frame ID (to call the SDK), pixel (x,y) coordinates
        (for imzML spatial index), and metadata (polarity, TIC, BPC).

        Coordinate resolution — tries three strategies in order:
          1. XIndexPos / YIndexPos  numeric grid indices (best, most common)
          2. SpotName               alphanumeric name parsed by _parse_spot_name
          3. Sequential fallback    x=1,2,...  y=1  (warns if used)
        """
        maldi_cols = []
        try:
            maldi_cols = [r[1] for r in conn.execute('PRAGMA table_info(MaldiFrameInfo)')]
        except Exception:
            pass

        # Only convert frames with detected ions and matching MS level.
        q = ('SELECT f.Id, f.Polarity, f.ScanMode, f.MsMsType, f.NumPeaks,'
             ' f.SummedIntensities, f.MaxIntensity'
             ' FROM Frames f WHERE f.SummedIntensities > 0')
        if self.ms_level_filter is not None:
            q += f' AND f.MsMsType = {self.ms_level_filter}'
        q += ' ORDER BY f.Id'
        rows = conn.execute(q).fetchall()

        coord_map = {}
        if 'XIndexPos' in maldi_cols and 'YIndexPos' in maldi_cols:
            for fid, x, y in conn.execute('SELECT Frame, XIndexPos, YIndexPos FROM MaldiFrameInfo'):
                coord_map[fid] = (int(x), int(y))
        elif 'SpotName' in maldi_cols:
            for fid, spot in conn.execute('SELECT Frame, SpotName FROM MaldiFrameInfo'):
                coord_map[fid] = _parse_spot_name(spot)
        else:
            print('WARNING: No pixel coordinates in MaldiFrameInfo — assigning sequential x.')
            for i, (fid, *_) in enumerate(rows):
                coord_map[fid] = (i + 1, 1)

        frames = []
        for (fid, polarity, scan_mode, msms_type, num_peaks, summed, maxint) in rows:
            x, y = coord_map.get(fid, (fid, 1))
            frames.append({'id': fid, 'x': x, 'y': y, 'polarity': polarity,
                           'scan_mode': scan_mode, 'msms_type': msms_type,
                           'num_peaks': num_peaks, 'tic': summed, 'bpc': maxint})
        return frames

    # ── Stage 2: Binary .ibd ────────────────────────────────

    def _write_ibd(self, lib, handle, frames):
        """
        Write all spectra to the .ibd binary file; record per-spectrum offsets.

        Processed-mode layout — for each pixel:
            [m/z array: N x float64]  [intensity array: N x float32]

        The byte offset of each block is recorded so the imzML XML can
        reference exactly where in the .ibd each spectrum lives.
        A running SHA-1 hash is computed over the whole file as we write;
        it is embedded in the imzML so tools can verify file integrity.
        """
        reader    = _read_centroid if self.spectrum_type == 'centroid' else _read_profile
        sha1      = hashlib.sha1()
        spec_meta = []
        progress  = (tqdm(total=len(frames), desc='Converting spectra', unit='px')
                     if HAS_TQDM else None)

        with open(self.ibd_path, 'wb') as f:
            byte_offset = 0

            for fr in frames:
                mzs, ints = reader(lib, handle, fr['id'])

                # m/z -> float64 (8 bytes): high precision for mass accuracy.
                # intensity -> float32 (4 bytes): sufficient for peak heights.
                mz_bytes  = mzs.astype(np.float64).tobytes()
                int_bytes = ints.astype(np.float32).tobytes()

                sha1.update(mz_bytes)
                sha1.update(int_bytes)

                mz_offset = byte_offset          # capture start BEFORE writing
                f.write(mz_bytes)
                byte_offset += len(mz_bytes)

                int_offset = byte_offset
                f.write(int_bytes)
                byte_offset += len(int_bytes)

                spec_meta.append({
                    'mz_offset':  mz_offset,  'mz_count':  len(mzs),
                    'int_offset': int_offset, 'int_count': len(ints),
                    'mz_min': float(mzs.min()) if len(mzs) else 0.0,
                    'mz_max': float(mzs.max()) if len(mzs) else 0.0,
                })

                if progress:
                    progress.update(1)

        if progress:
            progress.close()

        self._ibd_sha1 = sha1.hexdigest().upper()
        print(f'ibd SHA-1: {self._ibd_sha1}')
        return spec_meta

    # ── Stage 3: imzML XML ──────────────────────────────────

    def _write_imzml(self, meta, frames, spec_meta):
        """
        Write the .imzML XML file.

        Structure:
          cvList               declares which CV ontologies are used
          fileDescription      provenance + storage mode (processed)
          referenceableParams  reusable encoding declarations (avoids repetition)
          softwareList         what created this file
          scanSettingsList     imaging geometry (pixel grid, pixel size)
          instrumentConfig     MALDI source, TOF analyser, detector
          dataProcessingList   processing steps applied
          run/spectrumList     one <spectrum> per pixel: coords + .ibd pointers
        """
        xs, ys   = [f['x'] for f in frames], [f['y'] for f in frames]
        max_x, max_y = max(xs), max(ys)

        # Pixel size in µm. Try LaserWidth/Height from metadata; fall back to 100 µm.
        px = float(meta.get('MaldiFrameInfo.LaserWidth',  meta.get('PixelSizeX', 100.0)))
        py = float(meta.get('MaldiFrameInfo.LaserHeight', meta.get('PixelSizeY', px)))

        pol    = frames[0]['polarity']
        pol_cv = CV_POSITIVE if pol == '+' else CV_NEGATIVE
        pol_nm = 'positive scan' if pol == '+' else 'negative scan'

        sp_cv = CV_CENTROID if self.spectrum_type == 'centroid' else CV_PROFILE
        sp_nm = 'centroid spectrum' if self.spectrum_type == 'centroid' else 'profile spectrum'

        run_uuid = str(uuid.uuid4()).upper()

        # Build XML as a list of lines; join once at the end.
        # This avoids repeated string concatenation (O(n^2)) in the per-spectrum loop.
        L = []
        a = L.append

        a('<?xml version="1.0" encoding="utf-8"?>')
        a('<mzML xmlns="http://psi.hupo.org/ms/mzml" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"')
        a('      xsi:schemaLocation="http://psi.hupo.org/ms/mzml http://psidev.info/files/ms/mzML/xsd/mzML1.1.0_idx.xsd"')
        a('      xmlns:cv="http://psi.hupo.org/ms/mzml" version="1.1">')

        # cvList: declares which ontologies we reference.
        # Parsers use these URIs to look up CV term definitions.
        a('  <cvList count="3">')
        a('    <cv id="MS"  fullName="Proteomics Standards Initiative Mass Spectrometry Ontology" version="4.1.30" URI="https://raw.githubusercontent.com/HUPO-PSI/psi-ms-CV/master/psi-ms.obo"/>')
        a('    <cv id="IMS" fullName="Mass Spectrometry Imaging Ontology" version="1.1.0" URI="https://raw.githubusercontent.com/imzML/imzML/master/imagingMS.obo"/>')
        a('    <cv id="UO"  fullName="Unit Ontology" version="09:04:2014" URI="https://raw.githubusercontent.com/bio-ontology-research-group/unit-ontology/master/unit.obo"/>')
        a('  </cvList>')

        # fileDescription: declares storage mode (processed) and source provenance.
        a('  <fileDescription>')
        a('    <fileContent>')
        a(f'      <cvParam cvRef="MS"  accession="{sp_cv}" name="{sp_nm}" value=""/>')
        a(f'      <cvParam cvRef="IMS" accession="{CV_PROCESSED_MODE}" name="processed" value=""/>')
        a('    </fileContent>')
        a('    <sourceFileList count="1">')
        a(f'      <sourceFile id="sf1" name="{self.d_path.name}" location="{self.d_path.parent.as_uri()}">')
        a('        <cvParam cvRef="MS" accession="MS:1000564" name="PSI mzData file" value=""/>')
        a('      </sourceFile>')
        a('    </sourceFileList>')
        a('  </fileDescription>')

        # referenceableParamGroups: define binary encoding once; reuse by ID.
        # Without this, every <spectrum> would repeat the same four cvParam lines.
        a('  <referenceableParamGroupList count="2">')
        a('    <referenceableParamGroup id="mzArray">')
        a(f'      <cvParam cvRef="MS"  accession="{CV_MZ_ARRAY}"       name="m/z array"     value=""/>')
        a(f'      <cvParam cvRef="MS"  accession="{CV_64BIT_FLOAT}"    name="64-bit float"  value=""/>')
        a(f'      <cvParam cvRef="MS"  accession="{CV_NO_COMPRESSION}" name="no compression" value=""/>')
        a(f'      <cvParam cvRef="IMS" accession="{CV_EXTERNAL_DATA}"  name="external data"  value="true"/>')
        a('    </referenceableParamGroup>')
        a('    <referenceableParamGroup id="intensityArray">')
        a(f'      <cvParam cvRef="MS"  accession="{CV_INTENSITY_ARRAY}" name="intensity array" value=""/>')
        a(f'      <cvParam cvRef="MS"  accession="{CV_32BIT_FLOAT}"    name="32-bit float"   value=""/>')
        a(f'      <cvParam cvRef="MS"  accession="{CV_NO_COMPRESSION}" name="no compression"  value=""/>')
        a(f'      <cvParam cvRef="IMS" accession="{CV_EXTERNAL_DATA}"  name="external data"   value="true"/>')
        a('    </referenceableParamGroup>')
        a('  </referenceableParamGroupList>')

        a('  <softwareList count="1"><software id="bruker_to_imzml" version="1.0">')
        a('    <cvParam cvRef="MS" accession="MS:1000799" name="custom unreleased software tool" value="bruker_to_imzml"/>')
        a('  </software></softwareList>')

        # scanSettingsList: physical imaging geometry.
        a('  <scanSettingsList count="1"><scanSettings id="ss1">')
        a(f'    <cvParam cvRef="IMS" accession="{CV_MAX_COUNT_X}" name="max count of pixels x" value="{max_x}"/>')
        a(f'    <cvParam cvRef="IMS" accession="{CV_MAX_COUNT_Y}" name="max count of pixels y" value="{max_y}"/>')
        a(f'    <cvParam cvRef="IMS" accession="{CV_PIXEL_SIZE_X}" name="pixel size x" value="{px}" unitCvRef="UO" unitAccession="UO:0000017" unitName="micrometer"/>')
        a(f'    <cvParam cvRef="IMS" accession="{CV_PIXEL_SIZE_Y}" name="pixel size y" value="{py}" unitCvRef="UO" unitAccession="UO:0000017" unitName="micrometer"/>')
        a('  </scanSettings></scanSettingsList>')

        # instrumentConfiguration: MALDI source -> TOF analyser -> electron multiplier.
        a('  <instrumentConfigurationList count="1"><instrumentConfiguration id="IC1">')
        a('    <cvParam cvRef="MS" accession="MS:1001535" name="Bruker Daltonics timsTOF series" value=""/>')
        a('    <componentList count="3">')
        a(f'    <source order="1"><cvParam cvRef="MS" accession="{CV_MALDI}" name="matrix-assisted laser desorption ionization" value=""/></source>')
        a('    <analyzer order="2"><cvParam cvRef="MS" accession="MS:1000084" name="time-of-flight" value=""/></analyzer>')
        a('    <detector order="3"><cvParam cvRef="MS" accession="MS:1000253" name="electron multiplier" value=""/></detector>')
        a('    </componentList></instrumentConfiguration></instrumentConfigurationList>')

        # dataProcessingList: what was done to produce this file.
        a('  <dataProcessingList count="1"><dataProcessing id="dp1">')
        a('    <processingMethod order="1" softwareRef="bruker_to_imzml">')
        if self.spectrum_type == 'centroid':
            a('      <cvParam cvRef="MS" accession="MS:1000035" name="peak picking" value=""/>')
        a('      <cvParam cvRef="MS" accession="MS:1000544" name="Conversion to mzML" value=""/>')
        a('    </processingMethod></dataProcessing></dataProcessingList>')

        # run/spectrumList: one <spectrum> per pixel.
        a(f'  <run defaultInstrumentConfigurationRef="IC1" id="{run_uuid}">')
        a(f'    <spectrumList count="{len(frames)}" defaultDataProcessingRef="dp1">')

        for i, (fr, sm) in enumerate(zip(frames, spec_meta)):
            a(f'      <spectrum id="spectrum={i+1}" defaultArrayLength="{sm["mz_count"]}" index="{i+1}">')
            a(f'        <cvParam cvRef="MS" accession="MS:1000511" name="ms level" value="1"/>')
            a(f'        <cvParam cvRef="MS" accession="{sp_cv}" name="{sp_nm}" value=""/>')
            a(f'        <cvParam cvRef="MS" accession="{pol_cv}" name="{pol_nm}" value=""/>')
            a(f'        <cvParam cvRef="MS" accession="{CV_MZ_MIN}" name="lowest observed m/z"  value="{sm["mz_min"]:.6f}"/>')
            a(f'        <cvParam cvRef="MS" accession="{CV_MZ_MAX}" name="highest observed m/z" value="{sm["mz_max"]:.6f}"/>')
            # scanList: maps this spectrum to its position in the image grid.
            a('        <scanList count="1"><cvParam cvRef="MS" accession="MS:1000795" name="no combination" value=""/><scan>')
            a(f'          <cvParam cvRef="IMS" accession="{CV_PIXEL_COORD_X}" name="position x" value="{fr["x"]}"/>')
            a(f'          <cvParam cvRef="IMS" accession="{CV_PIXEL_COORD_Y}" name="position y" value="{fr["y"]}"/>')
            a('        </scan></scanList>')
            # binaryDataArrayList: pointers into the .ibd file.
            # external offset = byte start; external length = element count (not bytes).
            a('        <binaryDataArrayList count="2">')
            a('          <binaryDataArray><referenceableParamGroupRef ref="mzArray"/>')
            a(f'            <cvParam cvRef="IMS" accession="{CV_EXTERNAL_OFFSET}" name="external offset" value="{sm["mz_offset"]}"/>')
            a(f'            <cvParam cvRef="IMS" accession="{CV_EXTERNAL_LENGTH}" name="external length" value="{sm["mz_count"]}"/>')
            a('            <cvParam cvRef="MS" accession="MS:1000786" name="non-standard data array" value=""/><binary/></binaryDataArray>')
            a('          <binaryDataArray><referenceableParamGroupRef ref="intensityArray"/>')
            a(f'            <cvParam cvRef="IMS" accession="{CV_EXTERNAL_OFFSET}" name="external offset" value="{sm["int_offset"]}"/>')
            a(f'            <cvParam cvRef="IMS" accession="{CV_EXTERNAL_LENGTH}" name="external length" value="{sm["int_count"]}"/>')
            a('            <cvParam cvRef="MS" accession="MS:1000786" name="non-standard data array" value=""/><binary/></binaryDataArray>')
            a('        </binaryDataArrayList>')
            a('      </spectrum>')

        a('    </spectrumList></run></mzML>')

        with open(self.imzml_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(L))


print('BrukerToImzML class defined')
print()
print('All code loaded — proceed to Cell 5 to run the conversion.')

Imports OK
SDK loader defined
Spectrum readers defined
CV constants defined
Spot name parser defined
BrukerToImzML class defined

All code loaded — proceed to Cell 5 to run the conversion.


---
## Cell 5 — Run the conversion

Uses the settings from Cell 3. A progress bar appears if `tqdm` is installed.

> **Note:** If the `.ibd` file already exists from a previous run, delete it first — the writer appends rather than overwrites.

In [7]:
converter = BrukerToImzML(
    bruker_d_path    = BRUKER_D_PATH,
    output_dir       = OUTPUT_DIR,
    output_name      = OUTPUT_NAME,
    sdk_path         = SDK_PATH,
    spectrum_type    = SPECTRUM_TYPE,
    use_recalibrated = USE_RECALIBRATED,
    ms_level_filter  = MS_LEVEL_FILTER,
)

imzml_path, ibd_path = converter.convert()

Opening: C:\Users\maddi\Documents\ColleyVUApps\QTOFBrain\RatBrainNeg.d
Recalibrated state: available=False | using=False
Spectra to convert : 45927
Spectrum type      : centroid



Converting spectra: 100%|██████████| 45927/45927 [01:10<00:00, 653.35px/s]


ibd SHA-1: 3479731ECBD557FFC2361F377FBA35322C64A434

Done!
  imzML : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs\RatBrainNegConverted.imzML
  ibd   : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs\RatBrainNegConverted.ibd


---
## Cell 6 — Verify the output

Quick sanity checks: output file sizes and a dataset summary.

In [8]:
import os, sqlite3

print('=== Output files ===')
print(f'  imzML : {imzml_path}')
print(f'         {os.path.getsize(imzml_path) / 1024:.1f} KB')
print(f'  ibd   : {ibd_path}')
print(f'         {os.path.getsize(ibd_path) / 1024 / 1024:.1f} MB')
print()

# Quick dataset summary from the SQLite database
conn = sqlite3.connect(str(converter.d_path / 'analysis.tsf'))

n_total = conn.execute('SELECT COUNT(*) FROM Frames').fetchone()[0]
n_data  = conn.execute('SELECT COUNT(*) FROM Frames WHERE SummedIntensities > 0').fetchone()[0]

try:
    xs = [r[0] for r in conn.execute('SELECT XIndexPos FROM MaldiFrameInfo')]
    ys = [r[0] for r in conn.execute('SELECT YIndexPos FROM MaldiFrameInfo')]
    grid_info = f'{max(xs)} x {max(ys)} = {max(xs)*max(ys)} pixels'
except Exception:
    grid_info = '(coordinate info not available)'

conn.close()

print('=== Dataset summary ===')
print(f'  Total frames        : {n_total}')
print(f'  Frames with data    : {n_data}')
print(f'  Pixel grid          : {grid_info}')
print(f'  ibd SHA-1           : {converter._ibd_sha1}')
print()
print('Ready to open in Cardinal, MSiReader, or SCiLS.')

=== Output files ===
  imzML : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs\RatBrainNegConverted.imzML
         78054.3 KB
  ibd   : C:\Users\maddi\Documents\ColleyVUApps\imzMLConverter\outputs\RatBrainNegConverted.ibd
         8739.6 MB

=== Dataset summary ===
  Total frames        : 45927
  Frames with data    : 45927
  Pixel grid          : 3872 x 577 = 2234144 pixels
  ibd SHA-1           : 3479731ECBD557FFC2361F377FBA35322C64A434

Ready to open in Cardinal, MSiReader, or SCiLS.


---
## Troubleshooting

| Error | Likely cause | Fix |
|---|---|---|
| `OSError: libtimsdata.so: cannot open` | SDK library not found | Set `SDK_PATH` in Cell 3 |
| `No analysis.tsf found` | Wrong directory or TDF (ion mobility) dataset | Point to the `.d` directory itself |
| `No matching frames` | `MS_LEVEL_FILTER` excludes all frames | Try `MS_LEVEL_FILTER = None` |
| Output looks wrong | Partial run left a stale `.ibd` | Delete old `.ibd` and re-run |
| `Failed to open dataset` | Corrupted `.d` or wrong-platform SDK | Verify in FlexImaging / DataAnalysis |

---

### Inspect the SQLite database directly

```python
import sqlite3
conn = sqlite3.connect('your_dataset.d/analysis.tsf')

# List all tables
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())

# Global metadata
for row in conn.execute('SELECT * FROM GlobalMetadata LIMIT 20'):
    print(row)

# Frame count by MS level
for row in conn.execute('SELECT MsMsType, COUNT(*) FROM Frames GROUP BY MsMsType'):
    print(row)

conn.close()
```